# Semana 9 — SQL para Data Warehouse

Nesta semana, vamos sair da etapa de construção do modelo dimensional e usar o Star Schema para responder perguntas de BI com SQL.

## Objetivos da Semana 9

Ao final das 3 aulas, você deverá conseguir:

- Usar `SELECT`, `WHERE` e `ORDER BY` em consultas analíticas.
- Fazer `INNER JOIN` entre tabela fato e dimensões.
- Usar `LEFT JOIN` para investigar ausência de relacionamento.
- Criar métricas com `GROUP BY`, `SUM`, `COUNT` e `AVG`.
- Filtrar agregações com `HAVING`.
- Usar funções de data como `DATE_TRUNC` e `EXTRACT`.

## Aula 1 da Semana 9 — SQL básico em DW

### Roteiro teórico

Em um Data Warehouse, o SQL é usado principalmente para análise.  
Diferente de um sistema transacional, onde o foco é inserir e atualizar registros, aqui o foco é consultar, agregar e transformar dados em indicadores.

A estrutura básica de uma consulta é:

```sql
SELECT colunas
FROM tabela
WHERE condição
ORDER BY coluna;
```

No nosso modelo:

- `fato_vendas` guarda os eventos mensuráveis de venda.
- `dim_cliente` explica quem comprou e onde.
- `dim_produto` explica o que foi vendido.
- `dim_tempo` permite analisar a venda no tempo.

### Ler dados dimensão e fato

In [1]:
### Importar pacotes

import pandas as pd
import duckdb

In [2]:
dim_cliente = pd.read_csv('../Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv('../Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv('../Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv('../Dataset_schema/fato_vendas.csv')

In [3]:
banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
banco_dados.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
banco_dados.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
banco_dados.register('fato_vendas', fato_vendas) # registra a tabela fato vendas no banco de dados

## Consulta exemplo — Faturamento total

In [4]:
banco_dados.sql("""
SELECT SUM(valor_total) AS faturamento_total --  seleciona a soma da coluna valor_total da tabela fato_vendas e atribui o nome faturamento_total
FROM fato_vendas -- seleciona a tabela fato_vendas
""").df() # consulta para calcular o faturamento total da loja, somando a coluna valor_total da tabela fato_vendas

,faturamento_total
0,1.541977e+07


### Exemplo — selecionando vendas da fato

- `fato_vendas`: registra números e códigos toda vez que uma venda acontece.

Quero ver o identificador do produto (order_id), o valor do produto (valor_produto) e o valor do frete (valor_frete), dados que estão na tabela (fato_vendas).

In [5]:
banco_dados.sql("""
SELECT -- seleciona as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total da tabela fato_vendas (o que eu quero ver)
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas -- seleciona a tabela fato_vendas (de onde vem)
LIMIT 20 -- limita a consulta para mostrar apenas as 20 primeiras linhas (para não mostrar tudo)
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


### Exemplo — usando `WHERE` e `ORDER BY`

A consulta abaixo busca vendas com valor total maior que 200 e ordena da maior para a menor.

In [6]:
banco_dados.sql("""
SELECT -- seleciona as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total da tabela fato_vendas
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas -- seleciona a tabela fato_vendas
WHERE valor_total > 200 -- filtra os resultados para mostrar apenas as vendas com valor total maior que 200
ORDER BY valor_total DESC -- ordena os resultados pelo valor total em ordem decrescente
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6735.00,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
...,...,...,...,...,...
17622,31540,491e0cbf42a355f2797b69226d4db636,189.90,10.14,200.04
17623,106711,f7f3e95cf456d7a1ebede3e7c3cce324,161.42,38.60,200.02
17624,18378,2b31af271f3efcfd1f240a6ba82c631e,161.42,38.60,200.02
17625,24976,3a1f0844864136aa18f6710238f40e6e,161.42,38.60,200.02


## Exercício 13 🟢 Fácil — SELECT em tabela fato

Liste as colunas `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total` das 15 primeiras vendas.

In [7]:
banco_dados.sql("""
    SELECT -- seleciona as colunas venda_sk, order_id, valor_produto, valor_frete e valor_total da tabela fato_vendas
        venda_sk,
        order_id,
        valor_produto,
        valor_frete,
        valor_total
    FROM fato_vendas -- seleciona a tabela fato_vendas
    LIMIT 15 -- limita a consulta para mostrar apenas as 15 primeiras linhas
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


## Exercício 14 🟢 Fácil — Filtro com WHERE

Liste as vendas cujo `valor_frete` seja maior que 50.  
Mostre `venda_sk`, `order_id`, `valor_frete` e `valor_total`.

In [8]:
banco_dados.sql("""
    SELECT
        venda_sk,
        order_id,
        valor_produto,
        valor_frete,
        valor_total
    FROM fato_vendas
    WHERE valor_frete > 50
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
1,74,002b430ff89b3a24c31a1170acbbedea,199.99,65.56,265.55
2,83,0030ff924c38549807645976adeef2c0,225.00,67.24,292.24
3,97,0036757472ece3dde52fd4bfd929c90e,136.99,66.04,203.03
4,109,003edccf16bc5ec447f592913b3df2b4,14.00,50.85,64.85
...,...,...,...,...,...
4454,109909,ff3e501f56dcf0752578d86df833558f,299.90,170.11,470.01
4455,109998,ff85f6534f8a6b89e27a340dcf86ecac,259.99,84.52,344.51
4456,110020,ff96d596c25445650eee60b94fa62244,329.00,127.55,456.55
4457,110101,ffc2638415f3ce34e88641eef792c1fc,629.00,67.08,696.08


## Exercício 15 🟡 Médio — Ordenação para ranking

Liste as 20 vendas de maior `valor_total`.  
Mostre `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total`.

In [9]:
banco_dados.sql("""
    SELECT
        venda_sk,
        order_id,
        valor_produto,
        valor_frete,
        valor_total
    FROM fato_vendas
    ORDER BY valor_total DESC
    LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6735.00,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,4590.00,91.78,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,4399.87,113.45,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,3999.00,195.76,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,4099.99,75.27,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,4059.00,104.51,4163.51


##  JOINS em Star Schema

### Roteiro teórico

Em um Star Schema, normalmente a análise começa na tabela fato e depois se conecta às dimensões.

Exemplo de raciocínio:

> Quero faturamento por estado.  
> O faturamento está na `fato_vendas`.  
> O estado está na `dim_cliente`.  
> Então preciso fazer JOIN entre fato e dimensão cliente.

Tipos importantes:

- `INNER JOIN`: retorna apenas registros que possuem correspondência nos dois lados.
- `LEFT JOIN`: mantém todos os registros da tabela da esquerda, mesmo sem correspondência na direita.

### Exemplo — fato + dimensão cliente

- Pega a tabela de clientes e junta as duas em uma tabelona só, mas apenas onde o código do cliente bater certinho nos dois lados.

     As tabelas vão se conectar através da coluna cliente_sk que é semelhante entre ambas (fato_vendas e dim_clientes).

In [10]:
banco_dados.sql("""
SELECT
    f.venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas
    f.order_id, -- seleciona a coluna order_id da tabela fato_vendas
    f.valor_total, -- seleciona a coluna valor_total da tabela fato_vendas
    c.customer_state -- seleciona a coluna customer_state da tabela dim_cliente
FROM fato_vendas f -- seleciona a tabela fato_vendas e atribui o nome f para ela
INNER JOIN dim_cliente c -- seleciona a tabela dim_cliente e atribui o nome c para ela, fazendo uma junção interna entre as duas tabelas
    ON f.cliente_sk = c.cliente_sk -- faz a junção entre as tabelas fato_vendas e dim_cliente usando a coluna cliente_sk que é a coluna comum entre as duas tabelas
LIMIT 10
""").df()

,venda_sk,order_id,valor_total,customer_state
0,1,00010242fe8c5a6d1ba2dd792cb16214,72.19,RJ
1,2,00018f77f2f0320c557190d7a144bdd3,259.83,SP
2,3,000229ec398224ef6ca0657da4fc703e,216.87,MG
3,4,00024acbcdf0a6daa1e931b038114c75,25.78,SP
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,SP
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,MG
6,7,00054e8431b9d7675808bcb819fb4a32,31.75,SP
7,8,000576fe39319847cbb9d288c5617fa6,880.75,SP
8,9,0005a1a1728c9d785b8e2b08b904576c,157.60,SP
9,10,0005f50442cb953dcd1d21e1fb923495,65.39,SP


### Exemplo — fato + dimensão produto + dimensão tempo

- Relatório detalhado das vendas onde possa ver o código da venda, o número do pedido, o nome da categoria do produto (em inglês), o mês em que a venda ocorreu, o valor total que foi pago e o estado de onde o cliente comprou.

In [11]:
banco_dados.sql("""
SELECT
    f.venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas e atribui o nome f para ela 
    f.order_id, -- seleciona a coluna order_id da tabela fato_vendas e atribui o nome f para ela
    p.product_category_name_english AS categoria, -- seleciona a coluna product_category_name_english da tabela dim_produto e atribui o nome categoria para ela
    t.ano_mes, -- seleciona a coluna ano_mes da tabela dim_tempo
    f.valor_total, -- seleciona a coluna valor_total da tabela fato_vendas
    c.customer_state AS estado -- seleciona a coluna customer_state da tabela dim_cliente e atribui o nome estado para ela
FROM fato_vendas f -- seleciona a tabela fato_vendas e atribui o nome f para ela
INNER JOIN dim_produto p -- seleciona a tabela dim_produto e atribui o nome p para ela, fazendo uma junção interna entre as duas tabelas
    ON f.produto_sk = p.produto_sk -- faz a junção entre as tabelas fato_vendas e dim_produto usando a coluna produto_sk que é a coluna comum entre as duas tabelas
INNER JOIN dim_tempo t -- seleciona a tabela dim_tempo e atribui o nome t para ela, fazendo uma junção interna entre as duas tabelas
    ON f.tempo_sk = t.tempo_sk -- faz a junção entre as tabelas fato_vendas e dim_tempo usando a coluna tempo_sk que é a coluna comum entre as duas tabelas
INNER JOIN dim_cliente c -- seleciona a tabela dim_cliente e atribui o nome c para ela, fazendo uma junção interna entre as duas tabelas
    ON f.cliente_sk = c.cliente_sk -- faz a junção entre as tabelas fato_vendas e dim_cliente usando a coluna cliente_sk que é a coluna comum entre as duas tabelas
LIMIT 20
""").df()

,venda_sk,order_id,categoria,ano_mes,valor_total,estado
0,102401,edc438144eedf58c321995efd0225e19,construction_tools_construction,2018-07,68.45,GO
1,102402,edc531c3153d23614049077e7e1e6405,perfumery,2018-04,136.40,MG
2,102403,edc577111e3237868aec11a4da6611d9,health_beauty,2018-07,61.58,MG
3,102404,edc5c94770796f4212d272830dd17840,cool_stuff,2018-01,42.58,SP
4,102405,edc6d3b93ee7050db4013694b93176ae,sports_leisure,2017-05,30.86,SP
5,102406,edc7444663af179dd113423c0512b4e4,bed_bath_table,2018-02,67.98,SP
6,102407,edc75396403e226e7ac4e42085860602,baby,2017-05,185.93,CE
7,102408,edc852e0d271566f7923b10c089c30ec,stationery,2018-08,133.42,RS
8,102409,edc89efa4164df7e21f701a9fb88bc98,fashion_bags_accessories,2017-11,55.15,SP
9,102410,edc89efa4164df7e21f701a9fb88bc98,fashion_bags_accessories,2017-11,73.00,SP


In [12]:
banco_dados.sql("""
-- esta análise tem como objetivo identificar os 10 pedidos com maior valor total, juntamente com as informações do cliente e do estado de entrega. 
-- A consulta utiliza uma junção entre a tabela fato_vendas e a tabela dim_cliente para obter os detalhes do cliente, e ordena os resultados pelo valor total em ordem decrescente, limitando a exibição aos 10 primeiros registros.
-- Relatórios financeiros (como descobrir o faturamento ou os maiores pedidos), o LEFT JOIN é o mais indicado por uma questão de segurança de negócios.       
SELECT -- seleciona as colunas venda_sk, order_id, cliente_sk, customer_city, customer_state e valor_total das tabelas fato_vendas e dim_cliente
    v.venda_sk, -- seleciona a coluna venda_sk da tabela fato_vendas e atribui o nome v para ela
    v.order_id, -- seleciona a coluna order_id da tabela fato_vendas e atribui o nome v para ela
    v.cliente_sk,
    c.customer_city, -- seleciona a coluna customer_city da tabela dim_cliente e atribui o nome c para ela
    c.customer_state, -- seleciona a coluna customer_state da tabela dim_cliente e atribui o nome c para ela
    v.valor_total
FROM fato_vendas v -- seleciona a tabela fato_vendas e atribui o nome v para ela
    LEFT JOIN dim_cliente c -- seleciona a tabela dim_cliente e atribui o nome c para ela, fazendo uma junção à esquerda entre as duas tabelas
ON v.cliente_sk = c.cliente_sk -- faz a junção entre as tabelas fato_vendas e dim_cliente usando a coluna cliente_sk que é a coluna comum entre as duas tabelas
    ORDER BY v.valor_total ASC 
 LIMIT 10;
 """).df()

,venda_sk,order_id,cliente_sk,customer_city,customer_state,valor_total
0,61577,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
1,61573,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
2,61574,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
3,61575,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
4,48999,71e22e2d99081d6dc07d9627bb85969e,69939,sao paulo,SP,7.28
5,56044,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
6,56039,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
7,56042,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
8,56041,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
9,56043,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09


# AULA 2 - SEMANA 9

Agregação de tabelas:
- GROUP_BY, COUNT, SUM e AVG
- WHERE vs HAVING

#### Exemplo 1 (slide 18) - Faturamento por estado

- faturamento total, a quantidade de pedidos feitos e o valor médio gasto por item em cada estado, ordenando do que mais vendeu para o que menos vendeu.

In [26]:
banco_dados.sql("""
    SELECT
        c.customer_state AS estado, -- seleciona a coluna customer_state da tabela dim_cliente e atribui o nome estado para ela
        SUM(f.valor_total) AS faturamento_total, -- soma a coluna valor_total da tabela fato_vendas e atribui o nome faturamento_total para ela
        COUNT(f.order_id) AS qtd_pedidos, -- Conta quantas linhas existem para cada estado, ou seja, quantos pedidos foram feitos em cada estado, e atribui o nome qtd_pedidos para essa contagem
        AVG(f.valor_total) AS valor_medio_item -- calcula a média de valor gasto por pedido para cada estado, e atribui o nome valor_medio_item para essa média
    FROM fato_vendas f
    INNER JOIN dim_cliente c 
        ON f.cliente_sk = c.cliente_sk 
    GROUP BY c.customer_state  -- agrupa os resultados pela coluna customer_state da tabela dim_cliente para obter os resultados por estado
    ORDER BY faturamento_total DESC;
""").df()
   

,estado,faturamento_total,qtd_pedidos,valor_medio_item
0,SP,5769703.15,46448,124.218549
1,RJ,2055401.57,14143,145.329956
2,MG,1818891.67,12916,140.824688
3,RS,861472.79,6134,140.442255
4,PR,781708.80,5649,138.380032
5,SC,595127.78,4097,145.259404
6,BA,591137.81,3683,160.504428
7,DF,346123.35,2355,146.973822
8,GO,334212.35,2277,146.777492
9,ES,317657.93,2225,142.767609


#### Exemplo 2 (slide 19) - Faturamento por categoria

- Descubra quais categorias de produtos dão mais dinheiro para a empresa

In [14]:
banco_dados.sql("""
    SELECT
        p.product_category_name_english AS categoria,
        SUM(f.valor_total) AS faturamento_total,
        COUNT(*) AS qtd_itens, -- conta quantas linhas existem para cada categoria, ou seja, quantos itens foram vendidos em cada categoria, e atribui o nome qtd_itens para essa contagem
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english  -- agrupa os resultados pela coluna product_category_name_english da tabela dim_produto para obter os resultados por categoria
    ORDER BY faturamento_total DESC;
 """).df()

,categoria,faturamento_total,qtd_itens,ticket_medio
0,health_beauty,1412089.53,9465,149.190653
1,watches_gifts,1264333.12,5859,215.793330
2,bed_bath_table,1225209.26,10953,111.860610
3,sports_leisure,1118256.91,8431,132.636331
4,computers_accessories,1032723.77,7644,135.102534
...,...,...,...,...
67,flowers,1598.91,33,48.451818
68,home_comfort_2,1170.58,30,39.019333
69,cds_dvds_musicals,954.99,14,68.213571
70,fashion_childrens_clothes,598.67,7,85.524286


### EXERCÍCIO (SLIDE 20) - Métricas por mês

In [15]:
banco_dados.sql("""
    SELECT
        t.ano_mes AS periodo, -- seleciona a coluna ano_mes da tabela dim_tempo e atribui o nome periodo para ela
        SUM(f.valor_total) AS faturamento_total, -- soma de tudo que entrou no caixa, ou seja, a soma da coluna valor_total da tabela fato_vendas e atribui o nome faturamento_total para ela
        COUNT(*) AS qnt_pedidos, -- total de tens ou pedidos vendidos no mês, e atribui o nome qnt_pedidos para essa contagem
        AVG(f.valor_total) AS ticket_medio -- quanto em média cada cliente gastou por pedido, e atribui o nome ticket_medio para essa média
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes  -- agrupa os resultados pela coluna ano_mes da tabela dim_tempo para obter os resultados por período
    ORDER BY t.ano_mes ASC; -- ordena os resultados pela coluna ano_mes da tabela dim_tempo em ordem crescente para mostrar os períodos do mais antigo para o mais recente
""").df()

,periodo,faturamento_total,qnt_pedidos,ticket_medio
0,2016-09,143.46,3,47.820000
1,2016-10,46490.66,313,148.532460
2,2016-12,19.62,1,19.620000
3,2017-01,127482.37,913,139.630197
4,2017-02,271239.32,1858,145.984564
5,2017-03,414330.95,2897,143.020694
6,2017-04,390812.40,2569,152.126275
7,2017-05,566851.40,4004,141.571279
8,2017-06,490050.37,3489,140.455824
9,2017-07,566299.08,4416,128.238016


## Exercício 16  — Faturamento por estado

Retorne o valor_total por estado
Dica: use as funções SUM, COUNT, AVG, GROUP_BY e ORDER_BY e JOIN

### 📊 Guia Rápido de Funções de Agregação (SQL)

| Se o pedido contiver as palavras: | O seu cérebro escolhe: | O que a função faz: |
| :--- | :--- | :--- |
| **"Faturamento", "Total", "Soma", "Acumulado"** | `SUM(coluna)` | Soma todos os valores numéricos da lista. |
| **"Quantidade", "Volume", "Total de pedidos", "Quantos"** | `COUNT(coluna)` | Conta o número de linhas (registros) que entram no grupo. |
| **"Média", "Ticket Médio", "Valor Médio"** | `AVG(coluna)` | Soma tudo e divide pela quantidade (Média Aritmética). |
| **"Maior valor", "Preço máximo", "Pico"** | `MAX(coluna)` | Vasculha a lista e encontra o maior número de todos. |
| **"Menor valor", "Preço mínimo"** | `MIN(coluna)` | Vasculha a lista e encontra o menor número de todos. |

> 💡 **Dica de Ouro:** Lembre-se que sempre que você usar qualquer uma dessas funções matemáticas junto com uma coluna comum no seu `SELECT`, você é **obrigado** a repetir a coluna comum lá embaixo na cláusula `GROUP BY`.

In [16]:
## Resposta exercício 16

banco_dados.sql("""
SELECT
    c.customer_state AS estado, -- seleciona a coluna customer_state da tabela dim_cliente e atribui o nome estado para ela
    SUM(f.valor_total) AS faturamento_total, -- soma a coluna valor_total da tabela fato_vendas e atribui o nome faturamento_total para ela
    COUNT(f.order_id) AS qtd_pedidos, -- Conta quantas linhas existem para cada estado, ou seja, quantos itens foram vendidos em cada estado, e atribui o nome qtd_pedidos para essa contagem
    AVG(f.valor_total) AS valor_medio_item -- calcula a média de valor gasto por pedido para cada estado, e atribui o nome valor_medio_item para essa média
FROM fato_vendas f -- seleciona a tabela fato_vendas e atribui o nome f para ela
INNER JOIN dim_cliente c -- faz a junção entre as tabelas fato_vendas e dim_cliente
    ON f.cliente_sk = c.cliente_sk -- faz a junção entre as tabelas fato_vendas e dim_cliente usando a coluna cliente_sk que é a coluna comum entre as duas tabelas
GROUP BY c.customer_state  -- agrupa os resultados pela coluna customer_state da tabela dim_cliente para obter os resultados por estado
ORDER BY faturamento_total DESC; -- ordena os resultados pela coluna faturamento_total em ordem decrescente para mostrar os estados com maior faturamento primeiro
""").df()

,estado,faturamento_total,qtd_pedidos,valor_medio_item
0,SP,5769703.15,46448,124.218549
1,RJ,2055401.57,14143,145.329956
2,MG,1818891.67,12916,140.824688
3,RS,861472.79,6134,140.442255
4,PR,781708.80,5649,138.380032
5,SC,595127.78,4097,145.259404
6,BA,591137.81,3683,160.504428
7,DF,346123.35,2355,146.973822
8,GO,334212.35,2277,146.777492
9,ES,317657.93,2225,142.767609


## Exercício 17 — Top 10 categorias

Retorne uma tabela que mostre as top 10 categorias de maior faturamento total

In [17]:
## Resposta exercício 17

banco_dados.sql("""
SELECT
    p.product_category_name_english AS categoria, -- seleciona a categoria do produto da tabela dim_produto
    SUM(f.valor_total) AS faturamento_total, -- soma a coluna valor_total da tabela fato_vendas
        COUNT(*) AS qtd_itens, -- conta quantas linhas existem para cada categoria, ou seja, quantos itens foram vendidos em cada categoria
    AVG(f.valor_total) AS valor_medio_item -- calcula a média de valor gasto por pedido para cada categoria
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
    LIMIT 10; -- limita a consulta para mostrar apenas as 10 primeiras linhas, ou seja, as 10 categorias mais vendidas
""").df()

,categoria,faturamento_total,qtd_itens,valor_medio_item
0,construction_tools_construction,162382.01,916,177.272937
1,luggage_accessories,168778.12,1077,156.711346
2,signaling_and_security,27792.59,197,141.079137
3,art,27485.53,197,139.520457
4,industry_commerce_and_business,46079.34,265,173.884302
5,fashion_male_clothing,12514.95,125,100.119600
6,books_technical,22901.07,263,87.076312
7,food_drink,19337.77,269,71.887621
8,furniture_bedroom,23379.22,103,226.982718
9,books_imported,5150.94,57,90.367368


## Exercício 18 (Aula 2 semana 9) — Os 5 estados com maior ticket médio

Retorne uma tabela que mostre o ticket médio por estado
Dica: Use a função COUNT, SUM, AVG e HAVING

In [22]:
banco_dados.sql("""
SELECT
    c.customer_state AS estado, 
    SUM(f.valor_total) AS faturamento_total, 
    COUNT(*) AS qtd_pedidos,
    AVG(f.valor_total) AS valor_medio_item
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY c.customer_state
HAVING SUM(f.valor_total) > 100 
ORDER BY valor_medio_item DESC
LIMIT 5; 
""").df()

,estado,faturamento_total,qtd_pedidos,valor_medio_item
0,PB,137838.55,586,235.219369
1,AL,94172.49,427,220.544473
2,AC,19575.33,91,215.113516
3,RO,56966.00,273,208.666667
4,PA,212023.57,1054,201.160882


## Exercício 19 (Aula 2 semana 9) — Evolução mensal de faturamento

Use função SUM, GROUP_BY, ORDER_BY E JOIN

In [24]:
### Resposta exercício 19

banco_dados.sql("""
SELECT
    t.ano_mes AS periodo,
    SUM(f.valor_total) AS faturamento_total,
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes
    ORDER BY t.ano_mes ASC;
""").df()

,periodo,faturamento_total
0,2016-09,143.46
1,2016-10,46490.66
2,2016-12,19.62
3,2017-01,127482.37
4,2017-02,271239.32
5,2017-03,414330.95
6,2017-04,390812.40
7,2017-05,566851.40
8,2017-06,490050.37
9,2017-07,566299.08


### EXERCÍCIOS SLIDE 27: PRATIQUE FILTROS DE GRUPO

#### Pedidos por estado

Enunciado: Liste os estados com mais de 500 pedidos únicos. Ordene do maior para o menor volume

In [27]:
banco_dados.sql("""
SELECT 
    c.customer_state AS estado,
    COUNT(*) AS volume_pedidos
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY c.customer_state
HAVING COUNT(*) > 500
ORDER BY volume_pedidos DESC;
""").df()

,estado,volume_pedidos
0,SP,46448
1,RJ,14143
2,MG,12916
3,RS,6134
4,PR,5649
5,SC,4097
6,BA,3683
7,DF,2355
8,GO,2277
9,ES,2225


#### Receita por categoria

Enunciado: Liste as categorias com receita total acima de R$ 50.000. Ordene pela receita.

In [28]:
banco_dados.sql("""
SELECT
    -- SELECIONA AS COLUNAS QUE SERÁO USADAS NO GRÁFICO
    p.product_category_name_english AS categoria,
    SUM(f.valor_total) AS receita_total,
    -- JUNÇÃO ENTRE AS TABELAS FATO_VENDAS E DIM_PRODUTO PARA OBTER A CATEGORIA DO PRODUTO
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
    HAVING SUM(f.valor_total) > 50000
    ORDER BY receita_total DESC;
""").df()

,categoria,receita_total
0,health_beauty,1412089.53
1,watches_gifts,1264333.12
2,bed_bath_table,1225209.26
3,sports_leisure,1118256.91
4,computers_accessories,1032723.77
5,furniture_decor,880329.92
6,housewares,758392.25
7,cool_stuff,691680.89
8,auto,669454.75
9,garden_tools,567145.68


#### Ticket médio por categoria

Enunciado: Use categoria como grupo e mostre apenas categorias com ticket médio acima de R$ 100.

In [29]:
banco_dados.sql("""
SELECT
    p.product_category_name_english AS categoria,
    AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
    HAVING AVG(f.valor_total) > 100
    ORDER BY ticket_medio DESC;
""").df()

,categoria,ticket_medio
0,computers,1147.486231
1,small_appliances_home_oven_and_coffee,674.601370
2,home_appliances_2,511.725195
3,agro_industry_and_commerce,369.918932
4,musical_instruments,310.579293
5,small_appliances,301.076702
6,fixed_telephony,234.574667
7,construction_tools_safety,232.262404
8,furniture_bedroom,226.982718
9,watches_gifts,215.793330


# AULA 3 - SEMANA 9
- DATE_TRUNC
- EXTRACT

### SLIDE 32: Transformando datas em período

In [30]:
banco_dados.sql("""
-- Conversão para data no SQL
    SELECT CAST(t.data AS DATE) AS data_pedido
        FROM dim_tempo t
        LIMIT 5
""").df()

,data_pedido
0,2017-10-02
1,2018-07-24
2,2018-08-08
3,2017-11-18
4,2018-02-13


In [31]:
banco_dados.sql("""
-- Agrupamento no início do mês
    SELECT 
        DATE_TRUNC('month', CAST(t.data AS DATE)) AS mes_referencia
        FROM dim_tempo t
        LIMIT 5
""").df()

,mes_referencia
0,2017-10-01
1,2018-07-01
2,2018-08-01
3,2017-11-01
4,2018-02-01


### Slide 33 - EXTRACT: Conceito e Sintaxe
Extraindo partes da data

In [32]:
banco_dados.sql("""
SELECT
    CAST(t.data AS DATE) AS data_pedido,
    EXTRACT(YEAR FROM CAST(t.data AS DATE)) AS ano,
    EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
    EXTRACT(QUARTER FROM CAST(t.data AS DATE)) AS trimestre,
    FROM dim_tempo t
    LIMIT 10
""").df()

,data_pedido,ano,mes,trimestre
0,2017-10-02,2017,10,4
1,2018-07-24,2018,7,3
2,2018-08-08,2018,8,3
3,2017-11-18,2017,11,4
4,2018-02-13,2018,2,1
5,2017-07-09,2017,7,3
6,2017-05-16,2017,5,2
7,2017-01-23,2017,1,1
8,2017-07-29,2017,7,3
9,2017-07-13,2017,7,3


### SLIDE 34 EXTRACT: Totais por Ano e Análise de Sazonalidade
Receita por ano e sazonalidade por mês

In [33]:
banco_dados.sql("""
    SELECT
        EXTRACT(YEAR FROM CAST(t.data AS DATE)) AS ano,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_anual
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY EXTRACT(YEAR FROM CAST(t.data AS DATE))
    ORDER BY ano ASC;
""").df()

,ano,qtd_pedidos,receita_anual
0,2016,317,46653.74
1,2017,49556,6921535.24
2,2018,60324,8451584.77


### Sazonalidade por mês

In [34]:
banco_dados.sql("""
-- Sazonalidade por mês   
    SELECT
        EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_mensal
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY 1 -- agrupa pelo número da coluna mes
    ORDER BY mes ASC;
""").df()

,mes,qtd_pedidos,receita_mensal
0,1,8950,1205369.83
1,2,9376,1237407.73
2,3,10914,1534929.19
3,4,10396,1523691.33
4,5,11814,1695625.92
5,6,10499,1502028.66
6,7,11379,1594106.36
7,8,11939,1631324.00
8,9,4740,701220.95
9,10,5527,797607.67


In [35]:
banco_dados.sql("""
    
    SELECT
        EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_mensal
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY 1 -- agrupa pelo número da coluna mes
    ORDER BY mes ASC;
""").df()

,mes,qtd_pedidos,receita_mensal
0,1,8950,1205369.83
1,2,9376,1237407.73
2,3,10914,1534929.19
3,4,10396,1523691.33
4,5,11814,1695625.92
5,6,10499,1502028.66
6,7,11379,1594106.36
7,8,11939,1631324.00
8,9,4740,701220.95
9,10,5527,797607.67
